# Config

In [27]:
from pathlib import Path 

import numpy as np
import pandas as pd 
import polars as pl

# /kaggle/input/ts-prediction/train.parquet
# /kaggle/input/ts-prediction/test.parquet

DATA_DIR = Path("/kaggle/input/ts-prediction")

assert DATA_DIR.exists() and DATA_DIR.is_dir(), "Could not find input directory"

TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"

assert TRAIN_PATH.exists() and TEST_PATH.exists(), "Expected both train and test file paths to exist but at least one is missing from mounted kernel."

SAMPLE_SIZE = 300_000

HORIZON_MAP = {
    1: 'short',
    3: "medium",
    10: "long_short",
    25: "long_long"
}


# 23 entity unique codes (code)
# 47 entity segments (sub_code)
# 5 parent categorical labels describing where the entity belongs (sub_category)
cols = ['feature_a','feature_b','feature_c','feature_d','feature_e','feature_f','feature_g','feature_h','feature_i','feature_j','feature_k',
 'feature_l','feature_m','feature_n','feature_o','feature_p','feature_q','feature_r','feature_s','feature_t','feature_u','feature_v','feature_w',
 'feature_x','feature_y','feature_z','feature_aa','feature_ab','feature_ac','feature_ad','feature_ae','feature_af','feature_ag','feature_ah','feature_ai',
 'feature_aj','feature_ak','feature_al','feature_am','feature_an','feature_ao','feature_ap','feature_aq','feature_ar','feature_as','feature_at',
 'feature_au','feature_av','feature_aw','feature_ax','feature_ay','feature_az','feature_ba','feature_bb','feature_bc','feature_bd','feature_be',
 'feature_bf','feature_bg','feature_bh','feature_bi','feature_bj','feature_bk','feature_bl','feature_bm','feature_bn','feature_bo','feature_bp',
 'feature_bq','feature_br','feature_bs','feature_bt','feature_bu','feature_bv','feature_bw','feature_bx','feature_by','feature_bz','feature_ca',
 'feature_cb','feature_cc','feature_cd','feature_ce','feature_cf','feature_cg','feature_ch']

class CollarTag:
    """
        Example usage:

            ```python
            # (Encode Columns - Optional)

            df.with_columns(
                [
                    pl.col('sub_category')
                    .replace(SubCat.tag2id)
                    .cast(pl.Int32),
                    pl.col('sub_code')
                    .replace(SubCodeCat.tag2id)
                    .cast(pl.Int32),
                    pl.col('code')
                    .replace(CodeCat.tag2id)
                    .cast(pl.Int32)
                ]
            )
            ```
    """
    def __init__(self, inputs: list[str] | dict):

        if isinstance(inputs, dict): 
            print('Received dict type to encode. Encoding based on the assumption that it is HORIZON_MAP. If not, pls update this code.')
            self.labels = sorted(list(inputs.values()), reverse=False)
            self.id2tag = inputs
            self.tag2id = {tag: idx for idx, tag in inputs.items()}
        else:
            if not isinstance(inputs, list):
                print(f'Expected inputs to be list but received {type(inputs)}. Converting to list now')
                inputs = list(inputs)
                
            self.labels = sorted(inputs, reverse=False)
            self.id2tag: dict = {idx: tag for idx, tag in enumerate(self.labels)}
            self.tag2id: dict = {tag: idx for idx, tag in enumerate(self.labels)}

#CodeCat = CollarTag(df['code'].unique())
#SubCodeCat = CollarTag(df['sub_code'].unique())
#SubCat = CollarTag(df['sub_category'].unique())
#HorizonCat = CollarTag(HORIZON_MAP)

group_cols=['code', 'sub_code', 'sub_category']


# Utils

In [28]:
# Evaluation Functions
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

def evaluate(y_target, y_pred, w):
    score = weighted_rmse_score(y_target, y_pred, w)
    residue = y_target - y_pred 
    try:
        return {
            "weighted_rmse_score": score, 
            "rmse": np.sqrt(np.mean(residue**2)).round(4).item(),
            "bias^2": (residue.mean()**2).round(4).item(),
            "var": residue.var().round(4).item(),
            "wrmse_rmse": np.sqrt(np.sum(w * residue**2)/np.sum(w)).round(4).item()
        }
    except Exception as e: 
        print(f"Error captured: {e}")
        return {
            "weighted_rmse_score": score
        }

# Feature Processors

In [29]:
def add_temporal_features(
    df: pl.DataFrame,
    group_cols: list[str] = ['code', 'sub_code', 'sub_category'],
    phase_cuts: list[float] = [0.3, 0.5, 0.7],
    ts_phase_tag: CollarTag | None = None
) -> tuple[pl.DataFrame, CollarTag, list[str]]:  # also returns spline col names
    """Adds ts_rank, ts_rank_norm, ts_sin, ts_cos, ts_phase + spline basis cols."""

    df = df.sort(group_cols + ['ts_index'])

    df = df.with_columns(
        pl.col('ts_index').rank("dense").over(group_cols).alias("ts_rank")
    ).with_columns(
        (pl.col('ts_rank') / pl.col('ts_rank').max().over(group_cols)).alias('ts_rank_norm')
    ).with_columns([
        (pl.col('ts_rank_norm') * 2 * np.pi).sin().alias('ts_sin'),
        (pl.col('ts_rank_norm') * 2 * np.pi).cos().alias('ts_cos'),
        pl.col('ts_rank_norm').cut(phase_cuts).alias('ts_phase'),
    ])

    if ts_phase_tag is None:
        ts_phase_tag = CollarTag(df['ts_phase'].unique())

    df = df.with_columns(
        pl.col('ts_phase').cast(pl.String).replace(ts_phase_tag.tag2id).cast(pl.Int32)
    )

    assert set(['ts_index', 'ts_rank', 'ts_rank_norm', 'ts_sin', 'ts_cos', 'ts_phase']).issubset(set(df.columns))

    return df, ts_phase_tag  # ← return spline col names

def preprocess(
    df: pl.DataFrame,
    cols: list[str] = cols,
    group_cols: list[str] = group_cols,
    phase_cuts: list[float] = [0.3, 0.5, 0.7],
    n_splines: int = 12,
    ts_phase_tag: CollarTag | None = None,  # pass fitted tag for test set
) -> tuple[pl.DataFrame, list[str], CollarTag]:
    """
    Full preprocessing pipeline.

    For train : call with ts_phase_tag=None  → fits and returns a new CollarTag
    For test  : call with ts_phase_tag=<fitted tag from train> → reuses train encoding
    """
    
    df, ts_phase_tag, spline_cols = add_temporal_features(
        df, group_cols, phase_cuts,
        ts_phase_tag=ts_phase_tag
    )
    print(f'Finished adding temporal features ({len(spline_cols)} spline cols)')

    feature_cols = cols

    df = (
        df
        .with_columns(pl.col(pl.NUMERIC_DTYPES).fill_nan(None))
        .with_columns(pl.col(feature_cols).fill_null(0.0))
    )
    print("Finished resolving nulls/nans")

    return df, feature_cols, ts_phase_tag

# Loads dataset
ds_train = pl.read_parquet(TRAIN_PATH)
ds_test = pl.read_parquet(TEST_PATH)

# Preprocesses dataset
df_train, feature_cols, ts_phase_tag = preprocess(ds_train, cols=cols) # type: ignore
df_test, _, _ = preprocess(ds_test,  cols=cols, ts_phase_tag=ts_phase_tag)

Expected inputs to be list but received <class 'polars.series.series.Series'>. Converting to list now
Finished adding temporal features (12 spline cols)
Finished resolving nulls/nans


/tmp/ipykernel_55/1958561292.py:79: DeprecationWarning: `NUMERIC_DTYPES` is deprecated. Define your own data type groups or use the `polars.selectors` module for selecting columns of a certain data type.
  .with_columns(pl.col(pl.NUMERIC_DTYPES).fill_nan(None))


Finished adding temporal features (12 spline cols)
Finished resolving nulls/nans


/tmp/ipykernel_55/1958561292.py:79: DeprecationWarning: `NUMERIC_DTYPES` is deprecated. Define your own data type groups or use the `polars.selectors` module for selecting columns of a certain data type.
  .with_columns(pl.col(pl.NUMERIC_DTYPES).fill_nan(None))


# LGBM Training

In [33]:
import numpy as np
from sklearn.base import clone
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, SplineTransformer
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.utils.validation")

SEED = 42

# ── Spline factory ────────────────────────────────────────────────────────────
def periodic_spline_transformer(period, n_splines=None, degree=3):
    if n_splines is None:
        n_splines = period
    n_knots = n_splines + 1
    return SplineTransformer(
        degree=degree,
        n_knots=n_knots,
        knots=np.linspace(0, period, n_knots).reshape(n_knots, 1),
        extrapolation="periodic",
        include_bias=True,
    )

# ts_rank_norm is in [0,1] mapped to [0, 2π], so period=1 on the normalised scale
SPLINE_PERIOD   = 1
SPLINE_N_SPLINES = 12   # tune: more splines = finer periodicity

# ── Columns ───────────────────────────────────────────────────────────────────
time_cols = ['ts_cos', 'ts_sin']   # replaced by spline expansion below

# ── Preprocessor factory (cloned per horizon) ─────────────────────────────────
def make_preprocessor(use_spline_pca: bool = False, pca_components: float = 0.95):
    if use_spline_pca:
        time_pipe = Pipeline([
            ("spline", ColumnTransformer([
                ("sp_cos", periodic_spline_transformer(SPLINE_PERIOD, SPLINE_N_SPLINES), [0]),
                ("sp_sin", periodic_spline_transformer(SPLINE_PERIOD, SPLINE_N_SPLINES), [1]),
            ])),
            ("scaler", StandardScaler()),
            ("pca",    PCA(n_components=pca_components, random_state=SEED)),
        ])
        time_transformer = ("time_spline_pca", time_pipe, time_cols)
    else:
        time_transformer = ("time_pass", "passthrough", time_cols)

    return ColumnTransformer(
        transformers=[
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), group_cols),
            ("num", "passthrough", feature_cols),
            time_transformer,
        ]
    ).set_output(transform="pandas")

selector_estimator = LGBMRegressor(n_estimators=500, n_jobs=-1, verbose=-1)
selector = SelectFromModel(estimator=selector_estimator, threshold="mean")

lgbm_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'feature_fraction': 0.9,
    'subsample': 0.75,
    'random_state': SEED,
    'n_estimators': 5000,
    'n_jobs': -1,
    'verbose': -1,
    'learning_rate': 0.065,
    'num_leaves': 296,
    'max_depth': 12,
    'min_child_samples': 88,
    'subsample': 0.7891691017340575,
    'colsample_bytree': 0.9037484715991748,
    'reg_alpha': 1.407892012906228,
    'reg_lambda': 7.212991559021246,
}


# ── Training loop ─────────────────────────────────────────────────────────────
models = {}
partitions = df_train.partition_by("horizon", as_dict=True)

HORIZON_CFG = {
    1:  {
        "train_phases":  [0],
        "eval_phase":    1,           
        "use_spline_pca": True,
        "pca_components": 0.98,
        "lgbm_overrides": {
            "num_leaves": 48,
            "max_depth": 10,
            "min_child_samples": 60,
            "learning_rate": 0.04,
            "n_estimators": 5000,
            "feature_fraction": 0.85,
            "subsample": 0.85,
            "reg_alpha": 2.0,
            "reg_lambda": 10.0,
        },
    },
    3:  {
        "train_phases":  [0],      
        "eval_phase":    1,         
        "use_spline_pca": True,
        "pca_components": 0.98,
        "lgbm_overrides": {
            "num_leaves": 64,
            "max_depth": 10,
            "min_child_samples": 70,
            "learning_rate": 0.04,
            "n_estimators": 5000,
            "feature_fraction": 0.85,
            "subsample": 0.85,
            "reg_alpha": 2.0,
            "reg_lambda": 10.0,
        },
    },
    10: {
        "train_phases":  [0, 1],  
        "eval_phase":    2,           
        "use_spline_pca": True,
        "pca_components": 0.98,
        "lgbm_overrides": {
            "num_leaves": 128,
            "max_depth": 8,
            "min_child_samples": 150,
            "learning_rate": 0.02,
            "n_estimators": 3000,
            "feature_fraction": 0.6,
            "subsample": 0.65,
            "reg_alpha": 3.0,
            "reg_lambda": 15.0,
        },
    },
    25: {
        "train_phases":  [0, 1, 2],
        "eval_phase":    3,
        "use_spline_pca": True,
        "pca_components": 0.98,
        "lgbm_overrides": {
            "num_leaves": 96,            # slightly up from 64 — model has room given 972k rows
            "max_depth": 7,
            "min_child_samples": 200,
            "learning_rate": 0.06,       # keep — convergence rate looks appropriate
            "n_estimators": 5000,        # current 5000 is fine, it'll early-stop ~3000
            "feature_fraction": 0.65,
            "subsample": 0.65,
            "reg_alpha": 4.0,
            "reg_lambda": 20.0,
        },
    },
}

# ── Training loop ──────────────────────────────────────────────────────────────
models     = {}
partitions = df_train.partition_by("horizon", as_dict=True)

for (h,), horizon_df in partitions.items():
    cfg = HORIZON_CFG[h]

    train = horizon_df.filter(pl.col("ts_phase").is_in(cfg["train_phases"]))
    eval_ = horizon_df.filter(pl.col("ts_phase") == cfg["eval_phase"])

    print(f"\nHorizon {h} | train rows: {len(train):,} | eval rows: {len(eval_):,}")

    X_train = train.select(group_cols + feature_cols + time_cols).to_pandas()
    X_eval  = eval_.select(group_cols + feature_cols + time_cols).to_pandas()
    y_train = train["y_target"].to_pandas()
    y_eval  = eval_["y_target"].to_pandas()
    w_train = train["weight"].to_pandas()
    w_eval  = eval_["weight"].to_pandas()

    # Preprocessor — spline+PCA for short/medium, passthrough for long
    preprocessor_h = make_preprocessor(
        use_spline_pca=cfg["use_spline_pca"],
        pca_components=cfg["pca_components"],
    )
    X_train_t = preprocessor_h.fit_transform(X_train)
    X_eval_t  = preprocessor_h.transform(X_eval)

    # Feature selection
    selector_h = clone(selector)
    selector_h.fit(X_train_t, y_train)
    X_train_s = selector_h.transform(X_train_t)
    X_eval_s  = selector_h.transform(X_eval_t)

    print(f"Horizon {h} ({'spline+PCA' if cfg['use_spline_pca'] else 'passthrough'}) | "
          f"Features: {X_train_s.shape[1]}/{X_train_t.shape[1]}")

    # Merge base params with horizon overrides
    params = {**lgbm_params, **cfg["lgbm_overrides"]}

    model = LGBMRegressor(**params)
    model.fit(
        X_train_s, y_train,
        sample_weight=w_train,
        eval_set=[(X_eval_s, y_eval)],
        eval_sample_weight=[w_eval],
        callbacks=[early_stopping(300, verbose=False), log_evaluation(100)],
    )

    models[f"h{h}"] = (preprocessor_h, selector_h, model)

    print(f"Horizon {h} | TRAIN:", evaluate(y_train.to_numpy(), model.predict(X_train_s), w_train.to_numpy()))
    print(f"Horizon {h} | EVAL: ", evaluate(y_eval.to_numpy(),  model.predict(X_eval_s),  w_eval.to_numpy()))


Horizon 3 | train rows: 39,018 | eval rows: 15,771
Horizon 3 (spline+PCA) | Features: 33/112
[100]	valid_0's rmse: 0.101543
[200]	valid_0's rmse: 0.0953054
[300]	valid_0's rmse: 0.0899765
[400]	valid_0's rmse: 0.0855181
[500]	valid_0's rmse: 0.0819071
[600]	valid_0's rmse: 0.0786951
[700]	valid_0's rmse: 0.0759557
[800]	valid_0's rmse: 0.0732699
[900]	valid_0's rmse: 0.0707004
[1000]	valid_0's rmse: 0.0685567
[1100]	valid_0's rmse: 0.0665076
[1200]	valid_0's rmse: 0.0645412
[1300]	valid_0's rmse: 0.0628032
[1400]	valid_0's rmse: 0.0610195
[1500]	valid_0's rmse: 0.0595386
[1600]	valid_0's rmse: 0.0580204
[1700]	valid_0's rmse: 0.0566416
[1800]	valid_0's rmse: 0.0552312
[1900]	valid_0's rmse: 0.0538575
[2000]	valid_0's rmse: 0.0523716
Horizon 3 | TRAIN: {'weighted_rmse_score': 0.8709911574874601, 'rmse': 0.1296, 'bias^2': 0.0, 'var': 0.0168, 'wrmse_rmse': 0.0583}
Horizon 3 | EVAL:  {'weighted_rmse_score': 0.8949484415906594, 'rmse': 0.1146, 'bias^2': 0.0, 'var': 0.0131, 'wrmse_rmse': 0.

/usr/local/lib/python3.12/dist-packages/lightgbm/callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


[100]	valid_0's rmse: 0.317904
[200]	valid_0's rmse: 0.312887
[300]	valid_0's rmse: 0.308481
[400]	valid_0's rmse: 0.305067
[500]	valid_0's rmse: 0.302186
[600]	valid_0's rmse: 0.300236
[700]	valid_0's rmse: 0.297722
[800]	valid_0's rmse: 0.294613
[900]	valid_0's rmse: 0.293093
[1000]	valid_0's rmse: 0.291183
[1100]	valid_0's rmse: 0.287671
[1200]	valid_0's rmse: 0.287023
[1300]	valid_0's rmse: 0.286893
[1400]	valid_0's rmse: 0.286298
[1500]	valid_0's rmse: 0.285528
[1600]	valid_0's rmse: 0.284258
[1700]	valid_0's rmse: 0.283863
[1800]	valid_0's rmse: 0.282931
[1900]	valid_0's rmse: 0.282709
[2000]	valid_0's rmse: 0.282112
[2100]	valid_0's rmse: 0.281449
[2200]	valid_0's rmse: 0.2803
[2300]	valid_0's rmse: 0.279298
[2400]	valid_0's rmse: 0.278715
[2500]	valid_0's rmse: 0.277928
[2600]	valid_0's rmse: 0.277173
[2700]	valid_0's rmse: 0.276944
[2800]	valid_0's rmse: 0.276246
[2900]	valid_0's rmse: 0.275663
[3000]	valid_0's rmse: 0.275254
[3100]	valid_0's rmse: 0.275051
[3200]	valid_0's rm

# Predictions

In [25]:
test_partitions = df_test.partition_by("horizon", as_dict=True)
predictions = []

for (h,), horizon_df in test_partitions.items():
    preprocessor_h, selector_h, model = models[f"h{h}"]

    X_test = horizon_df.select(group_cols + feature_cols + time_cols).to_pandas()

    # Apply same preprocessing pipeline as training
    X_test_t = preprocessor_h.transform(X_test)   # transform only, no fit
    X_test_s = selector_h.transform(X_test_t)

    preds = model.predict(X_test_s)

    predictions.append(
        horizon_df.select("id")
        .with_columns(pl.Series("prediction", preds))
    )
    
submission = pl.concat(predictions)

assert len(submission) == len(df_test), "Row count mismatch — check all horizons are predicted"
assert set(submission.columns) == {"id", "prediction"}, "Column mismatch"

submission.write_csv("submission.csv")
print(f"Submission saved | shape: {submission.shape}")
print(submission.head())

Submission saved | shape: (100000, 2)
shape: (5, 2)
┌─────────────────────────────────┬────────────┐
│ id                              ┆ prediction │
│ ---                             ┆ ---        │
│ str                             ┆ f64        │
╞═════════════════════════════════╪════════════╡
│ OSJL3A7Y__03W5PJSV__PZ9S1Z4V__… ┆ -0.003383  │
│ OSJL3A7Y__03W5PJSV__PZ9S1Z4V__… ┆ 0.168407   │
│ OSJL3A7Y__03W5PJSV__PZ9S1Z4V__… ┆ 0.143761   │
│ OSJL3A7Y__03W5PJSV__PZ9S1Z4V__… ┆ -0.016306  │
│ OSJL3A7Y__03W5PJSV__PZ9S1Z4V__… ┆ 0.21405    │
└─────────────────────────────────┴────────────┘
